In [2]:
# ============================================================
# Synthetic Digital Payment Dataset Generator
# ============================================================
#
# Project:
# Digital Payment Transaction Analytics using SQL + Power BI
#
# Dataset:
# - 20,000 customers
# - 2,000 merchants
# - 300,000 transactions
# - ~15,000-25,000 failed/pending transaction records
#
# Period:
# January 1, 2025 to December 31, 2025
#
# IMPORTANT:
# This is SYNTHETIC data.
# It does not represent real Paytm/UPI/customer data.
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

N_CUSTOMERS = 20_000
N_MERCHANTS = 2_000
N_TRANSACTIONS = 300_000

START_DATE = "2025-01-01"
END_DATE = "2025-12-31 23:59:59"

OUTPUT_DIR = "digital_payment_dataset"

rng = np.random.default_rng(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. MASTER LOCATION DATA
# ============================================================

# Major Indian cities and states.
# Population weights are approximate and are only used to
# create realistic-looking synthetic data.

locations = [
    ("Mumbai", "Maharashtra", 0.10),
    ("Delhi", "Delhi", 0.09),
    ("Bengaluru", "Karnataka", 0.09),
    ("Hyderabad", "Telangana", 0.07),
    ("Chennai", "Tamil Nadu", 0.06),
    ("Kolkata", "West Bengal", 0.06),
    ("Pune", "Maharashtra", 0.05),
    ("Ahmedabad", "Gujarat", 0.05),
    ("Jaipur", "Rajasthan", 0.04),
    ("Lucknow", "Uttar Pradesh", 0.04),
    ("Surat", "Gujarat", 0.04),
    ("Bhopal", "Madhya Pradesh", 0.03),
    ("Patna", "Bihar", 0.03),
    ("Bhubaneswar", "Odisha", 0.03),
    ("Chandigarh", "Chandigarh", 0.025),
    ("Coimbatore", "Tamil Nadu", 0.025),
    ("Indore", "Madhya Pradesh", 0.025),
    ("Nagpur", "Maharashtra", 0.025),
    ("Kochi", "Kerala", 0.025),
    ("Guwahati", "Assam", 0.02),
    ("Visakhapatnam", "Andhra Pradesh", 0.02),
    ("Ranchi", "Jharkhand", 0.015),
    ("Dehradun", "Uttarakhand", 0.015),
    ("Srinagar", "Jammu and Kashmir", 0.01),
]

location_df = pd.DataFrame(
    locations,
    columns=["city", "state", "weight"]
)

location_df["weight"] = (
    location_df["weight"] /
    location_df["weight"].sum()
)


# ============================================================
# 3. CUSTOMERS
# ============================================================

print("Generating customers...")

customer_ids = [
    f"CUST{i:06d}"
    for i in range(1, N_CUSTOMERS + 1)
]

customer_location_idx = rng.choice(
    len(location_df),
    size=N_CUSTOMERS,
    p=location_df["weight"].values
)

customer_city = location_df.iloc[
    customer_location_idx
]["city"].values

customer_state = location_df.iloc[
    customer_location_idx
]["state"].values


# ------------------------------------------------------------
# Age
# ------------------------------------------------------------

ages = np.clip(
    rng.normal(
        loc=34,
        scale=11,
        size=N_CUSTOMERS
    ).round(),
    18,
    70
).astype(int)


# ------------------------------------------------------------
# Gender
# ------------------------------------------------------------

genders = rng.choice(
    ["Male", "Female", "Other"],
    size=N_CUSTOMERS,
    p=[0.54, 0.44, 0.02]
)


# ------------------------------------------------------------
# Customer signup date
# ------------------------------------------------------------

signup_start = pd.Timestamp("2021-01-01")
signup_end = pd.Timestamp("2025-06-30")

signup_days = (
    signup_end - signup_start
).days

signup_dates = (
    signup_start +
    pd.to_timedelta(
        rng.integers(
            0,
            signup_days + 1,
            size=N_CUSTOMERS
        ),
        unit="D"
    )
)


# ------------------------------------------------------------
# Customer segment
# ------------------------------------------------------------

customer_segments = rng.choice(
    ["New", "Regular", "Premium"],
    size=N_CUSTOMERS,
    p=[0.20, 0.65, 0.15]
)


# ------------------------------------------------------------
# Preferred payment method
#
# Make payment preference depend partially on age.
# ------------------------------------------------------------

preferred_methods = []

for age in ages:

    if age < 25:
        probs = [0.65, 0.10, 0.08, 0.04, 0.13]

    elif age < 40:
        probs = [0.58, 0.14, 0.14, 0.07, 0.07]

    elif age < 55:
        probs = [0.45, 0.20, 0.20, 0.10, 0.05]

    else:
        probs = [0.32, 0.27, 0.23, 0.13, 0.05]

    preferred_methods.append(
        rng.choice(
            [
                "UPI",
                "Debit Card",
                "Credit Card",
                "Net Banking",
                "Wallet"
            ],
            p=probs
        )
    )


customers = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_name": [
        f"Customer_{i:06d}"
        for i in range(1, N_CUSTOMERS + 1)
    ],
    "age": ages,
    "gender": genders,
    "city": customer_city,
    "state": customer_state,
    "signup_date": signup_dates,
    "customer_segment": customer_segments,
    "preferred_payment_method": preferred_methods
})


# ============================================================
# 4. MERCHANTS
# ============================================================

print("Generating merchants...")

merchant_ids = [
    f"MER{i:05d}"
    for i in range(1, N_MERCHANTS + 1)
]


merchant_categories = [
    "Grocery",
    "Food",
    "Travel",
    "Electronics",
    "Fashion",
    "Healthcare",
    "Utilities",
    "Entertainment",
    "Education",
    "Other"
]

merchant_category_probs = [
    0.20,
    0.16,
    0.07,
    0.08,
    0.10,
    0.08,
    0.10,
    0.07,
    0.05,
    0.09
]


merchant_category = rng.choice(
    merchant_categories,
    size=N_MERCHANTS,
    p=merchant_category_probs
)


merchant_location_idx = rng.choice(
    len(location_df),
    size=N_MERCHANTS,
    p=location_df["weight"].values
)

merchant_city = location_df.iloc[
    merchant_location_idx
]["city"].values

merchant_state = location_df.iloc[
    merchant_location_idx
]["state"].values


merchant_type = rng.choice(
    ["Online", "Offline", "Hybrid"],
    size=N_MERCHANTS,
    p=[0.30, 0.45, 0.25]
)


merchant_onboarding_start = pd.Timestamp("2020-01-01")
merchant_onboarding_end = pd.Timestamp("2024-12-31")

merchant_days = (
    merchant_onboarding_end -
    merchant_onboarding_start
).days

merchant_onboarding_dates = (
    merchant_onboarding_start +
    pd.to_timedelta(
        rng.integers(
            0,
            merchant_days + 1,
            size=N_MERCHANTS
        ),
        unit="D"
    )
)


merchants = pd.DataFrame({
    "merchant_id": merchant_ids,
    "merchant_name": [
        f"Merchant_{i:05d}"
        for i in range(1, N_MERCHANTS + 1)
    ],
    "category": merchant_category,
    "city": merchant_city,
    "state": merchant_state,
    "merchant_type": merchant_type,
    "onboarding_date": merchant_onboarding_dates
})


# ============================================================
# 5. TRANSACTION GENERATION
# ============================================================

print("Generating transactions...")


# ------------------------------------------------------------
# Customer activity weights
#
# Premium customers generate more transactions.
# ------------------------------------------------------------

segment_activity = {
    "New": 0.7,
    "Regular": 1.0,
    "Premium": 2.2
}

customer_activity = np.array([
    segment_activity[s]
    for s in customer_segments
])

customer_activity = (
    customer_activity /
    customer_activity.sum()
)


transaction_customer_idx = rng.choice(
    N_CUSTOMERS,
    size=N_TRANSACTIONS,
    p=customer_activity
)


transaction_customer_ids = np.array(
    customer_ids
)[transaction_customer_idx]


# ------------------------------------------------------------
# Merchant activity weights
#
# Popular categories receive more transactions.
# ------------------------------------------------------------

category_activity = {
    "Grocery": 1.7,
    "Food": 1.6,
    "Travel": 0.7,
    "Electronics": 0.5,
    "Fashion": 0.8,
    "Healthcare": 0.7,
    "Utilities": 1.1,
    "Entertainment": 0.8,
    "Education": 0.5,
    "Other": 0.7
}

merchant_activity = np.array([
    category_activity[c]
    for c in merchant_category
])

merchant_activity = (
    merchant_activity /
    merchant_activity.sum()
)


transaction_merchant_idx = rng.choice(
    N_MERCHANTS,
    size=N_TRANSACTIONS,
    p=merchant_activity
)


transaction_merchant_ids = np.array(
    merchant_ids
)[transaction_merchant_idx]


transaction_merchant_categories = (
    np.array(merchant_category)
)[transaction_merchant_idx]


# ------------------------------------------------------------
# Transaction dates
#
# Generate dates with monthly seasonality.
# ------------------------------------------------------------

date_range = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="h"
)

# Monthly activity multiplier
month_weights = {
    1: 0.95,
    2: 0.92,
    3: 0.98,
    4: 0.96,
    5: 0.98,
    6: 1.00,
    7: 1.02,
    8: 1.00,
    9: 1.04,
    10: 1.15,
    11: 1.18,
    12: 1.25
}

date_months = date_range.month

date_weights = np.array([
    month_weights[m]
    for m in date_months
])


# ------------------------------------------------------------
# Hour-of-day weights
# ------------------------------------------------------------

hour_weights = np.array([
    0.15,  # 00
    0.10,  # 01
    0.07,  # 02
    0.05,  # 03
    0.04,  # 04
    0.04,  # 05
    0.06,  # 06
    0.10,  # 07
    0.20,  # 08
    0.35,  # 09
    0.45,  # 10
    0.55,  # 11
    0.70,  # 12
    0.65,  # 13
    0.55,  # 14
    0.50,  # 15
    0.55,  # 16
    0.65,  # 17
    0.80,  # 18
    0.95,  # 19
    1.00,  # 20
    0.90,  # 21
    0.65,  # 22
    0.35   # 23
])


# Create a weight for every hourly timestamp
hour_weight_array = np.array([
    hour_weights[h]
    for h in date_range.hour
])


combined_date_weights = (
    date_weights *
    hour_weight_array
)

combined_date_weights = (
    combined_date_weights /
    combined_date_weights.sum()
)


transaction_datetime = rng.choice(
    date_range,
    size=N_TRANSACTIONS,
    p=combined_date_weights
)


transaction_datetime = pd.to_datetime(
    transaction_datetime
)


# ------------------------------------------------------------
# Weekends
# Slightly increase food/entertainment/travel activity.
# ------------------------------------------------------------

weekday = transaction_datetime.dayofweek.values

is_weekend = weekday >= 5


# ------------------------------------------------------------
# Transaction type
# ------------------------------------------------------------

transaction_types = []

for category in transaction_merchant_categories:

    if category == "Utilities":
        probs = [0.15, 0.55, 0.10, 0.10, 0.10]

    elif category == "Food":
        probs = [0.80, 0.02, 0.03, 0.10, 0.05]

    elif category == "Travel":
        probs = [0.70, 0.02, 0.02, 0.18, 0.08]

    else:
        probs = [0.80, 0.05, 0.05, 0.07, 0.03]

    transaction_types.append(
        rng.choice(
            [
                "Purchase",
                "Bill Payment",
                "Recharge",
                "Subscription",
                "Transfer"
            ],
            p=probs
        )
    )


transaction_types = np.array(transaction_types)


# ------------------------------------------------------------
# Payment method
#
# Use customer's preferred method with probability,
# otherwise select another method.
# ------------------------------------------------------------

customer_preferred = np.array(
    preferred_methods
)[transaction_customer_idx]

payment_methods = []

method_list = [
    "UPI",
    "Debit Card",
    "Credit Card",
    "Net Banking",
    "Wallet"
]

for preferred in customer_preferred:

    if rng.random() < 0.72:
        payment_methods.append(preferred)
    else:
        payment_methods.append(
            rng.choice(method_list)
        )

payment_methods = np.array(payment_methods)


# ------------------------------------------------------------
# Transaction amount
#
# Different merchant categories have different
# spending distributions.
# ------------------------------------------------------------

category_amount_params = {

    "Grocery": (450, 0.70),
    "Food": (350, 0.65),
    "Travel": (3500, 0.80),
    "Electronics": (9000, 0.75),
    "Fashion": (1800, 0.70),
    "Healthcare": (2200, 0.80),
    "Utilities": (1200, 0.65),
    "Entertainment": (700, 0.65),
    "Education": (3000, 0.75),
    "Other": (1000, 0.70)
}


amounts = np.zeros(N_TRANSACTIONS)


for category, (median, sigma) in category_amount_params.items():

    mask = (
        transaction_merchant_categories ==
        category
    )

    count = mask.sum()

    if count > 0:

        amounts[mask] = rng.lognormal(
            mean=np.log(median),
            sigma=sigma,
            size=count
        )


# Keep amounts in a realistic range
amounts = np.clip(
    amounts,
    20,
    100_000
)

amounts = np.round(
    amounts,
    2
)


# ------------------------------------------------------------
# Premium customers spend more
# ------------------------------------------------------------

transaction_customer_segments = np.array(
    customer_segments
)[transaction_customer_idx]

premium_mask = (
    transaction_customer_segments ==
    "Premium"
)

new_mask = (
    transaction_customer_segments ==
    "New"
)

amounts[premium_mask] *= rng.uniform(
    1.10,
    1.35,
    premium_mask.sum()
)

amounts[new_mask] *= rng.uniform(
    0.85,
    1.00,
    new_mask.sum()
)

amounts = np.clip(
    amounts,
    20,
    100_000
)

amounts = np.round(
    amounts,
    2
)


# ------------------------------------------------------------
# Device type
#
# Younger customers use mobile devices more heavily.
# ------------------------------------------------------------

device_types = []

for age in ages[transaction_customer_idx]:

    if age < 30:
        probs = [0.78, 0.15, 0.07]

    elif age < 50:
        probs = [0.68, 0.20, 0.12]

    else:
        probs = [0.50, 0.28, 0.22]

    device_types.append(
        rng.choice(
            ["Android", "iOS", "Web"],
            p=probs
        )
    )

device_types = np.array(device_types)


# ------------------------------------------------------------
# City / state
#
# Most transactions happen in customer's city.
# Some transactions occur elsewhere.
# ------------------------------------------------------------

customer_cities = np.array(
    customer_city
)[transaction_customer_idx]

customer_states = np.array(
    customer_state
)[transaction_customer_idx]

transaction_cities = customer_cities.copy()
transaction_states = customer_states.copy()


different_location_mask = (
    rng.random(N_TRANSACTIONS) < 0.08
)

different_indices = np.where(
    different_location_mask
)[0]

if len(different_indices) > 0:

    random_location_idx = rng.choice(
        len(location_df),
        size=len(different_indices),
        p=location_df["weight"].values
    )

    transaction_cities[different_indices] = (
        location_df.iloc[random_location_idx]["city"].values
    )

    transaction_states[different_indices] = (
        location_df.iloc[random_location_idx]["state"].values
    )


# ============================================================
# 6. TRANSACTION STATUS
# ============================================================

# Base failure probabilities by payment method

failure_probability = {
    "UPI": 0.035,
    "Debit Card": 0.055,
    "Credit Card": 0.045,
    "Net Banking": 0.070,
    "Wallet": 0.050
}


base_failure_prob = np.array([
    failure_probability[m]
    for m in payment_methods
])


# Merchant category effect

category_failure_multiplier = {
    "Grocery": 0.95,
    "Food": 0.95,
    "Travel": 1.20,
    "Electronics": 1.15,
    "Fashion": 1.00,
    "Healthcare": 1.10,
    "Utilities": 1.05,
    "Entertainment": 1.00,
    "Education": 1.05,
    "Other": 1.00
}

failure_prob = base_failure_prob.copy()

for category, multiplier in category_failure_multiplier.items():

    mask = (
        transaction_merchant_categories ==
        category
    )

    failure_prob[mask] *= multiplier


# High-value transactions have slightly higher failure probability

high_value_mask = amounts > 20_000

failure_prob[high_value_mask] *= 1.15


# Weekend effect

failure_prob[is_weekend] *= 1.05


# Cap probabilities
failure_prob = np.clip(
    failure_prob,
    0,
    0.25
)


random_values = rng.random(
    N_TRANSACTIONS
)

failed_mask = (
    random_values <
    failure_prob
)


# Pending transactions
pending_probability = 0.012

pending_mask = (
    (~failed_mask) &
    (rng.random(N_TRANSACTIONS) < pending_probability)
)


transaction_status = np.full(
    N_TRANSACTIONS,
    "Success",
    dtype=object
)

transaction_status[failed_mask] = "Failed"

transaction_status[pending_mask] = "Pending"


# ============================================================
# 7. TRANSACTION DATAFRAME
# ============================================================

transactions = pd.DataFrame({

    "transaction_id": [
        f"TXN{i:08d}"
        for i in range(1, N_TRANSACTIONS + 1)
    ],

    "customer_id":
        transaction_customer_ids,

    "merchant_id":
        transaction_merchant_ids,

    "transaction_datetime":
        transaction_datetime,

    "amount":
        amounts,

    "payment_method":
        payment_methods,

    "transaction_status":
        transaction_status,

    "city":
        transaction_cities,

    "state":
        transaction_states,

    "device_type":
        device_types,

    "transaction_type":
        transaction_types
})


# ============================================================
# 8. TRANSACTION FAILURE TABLE
# ============================================================

print("Generating failure records...")


failure_indices = np.where(
    failed_mask
)[0]


failure_reasons = []


for idx in failure_indices:

    method = payment_methods[idx]

    if method == "UPI":

        reasons = [
            "Bank Server Error",
            "Network Error",
            "Insufficient Balance",
            "Payment Gateway Error",
            "Limit Exceeded"
        ]

        probs = [
            0.28,
            0.24,
            0.20,
            0.18,
            0.10
        ]

    elif method in ["Debit Card", "Credit Card"]:

        reasons = [
            "Insufficient Balance",
            "Invalid PIN",
            "Limit Exceeded",
            "Bank Server Error",
            "Payment Gateway Error"
        ]

        probs = [
            0.25,
            0.15,
            0.20,
            0.20,
            0.20
        ]

    elif method == "Net Banking":

        reasons = [
            "Bank Server Error",
            "Timeout",
            "Network Error",
            "Invalid PIN",
            "Limit Exceeded"
        ]

        probs = [
            0.30,
            0.25,
            0.20,
            0.10,
            0.15
        ]

    else:

        reasons = [
            "Insufficient Balance",
            "Network Error",
            "Payment Gateway Error",
            "Timeout",
            "Limit Exceeded"
        ]

        probs = [
            0.25,
            0.20,
            0.25,
            0.20,
            0.10
        ]

    failure_reasons.append(
        rng.choice(
            reasons,
            p=probs
        )
    )


failure_count = len(failure_indices)


retry_attempts = rng.choice(
    [1, 2, 3],
    size=failure_count,
    p=[0.70, 0.23, 0.07]
)


# Resolution status depends partly on retry attempt

resolution_statuses = []

for retry in retry_attempts:

    if retry == 1:
        probs = [0.55, 0.45]

    elif retry == 2:
        probs = [0.75, 0.25]

    else:
        probs = [0.88, 0.12]

    resolution_statuses.append(
        rng.choice(
            ["Resolved", "Unresolved"],
            p=probs
        )
    )


transaction_failures = pd.DataFrame({

    "failure_id": [
        f"FAIL{i:07d}"
        for i in range(1, failure_count + 1)
    ],

    "transaction_id":
        transactions.iloc[
            failure_indices
        ]["transaction_id"].values,

    "failure_reason":
        failure_reasons,

    "retry_attempt":
        retry_attempts,

    "resolution_status":
        resolution_statuses
})


# ============================================================
# 9. CLEAN DATA TYPES
# ============================================================

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"]
).dt.date

merchants["onboarding_date"] = pd.to_datetime(
    merchants["onboarding_date"]
).dt.date

transactions["transaction_datetime"] = pd.to_datetime(
    transactions["transaction_datetime"]
)


# ============================================================
# 10. SAVE CSV FILES
# ============================================================

print("\nSaving files...")

customers.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "customers.csv"
    ),
    index=False
)

merchants.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "merchants.csv"
    ),
    index=False
)

transactions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "transactions.csv"
    ),
    index=False
)

transaction_failures.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "transaction_failures.csv"
    ),
    index=False
)


# ============================================================
# 11. DATA QUALITY CHECKS
# ============================================================

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(
    f"Customers              : {len(customers):,}"
)

print(
    f"Merchants              : {len(merchants):,}"
)

print(
    f"Transactions           : {len(transactions):,}"
)

print(
    f"Failure records        : {len(transaction_failures):,}"
)


print("\nTransaction status:")

print(
    transactions[
        "transaction_status"
    ].value_counts()
)


print("\nPayment method:")

print(
    transactions[
        "payment_method"
    ].value_counts()
)


print("\nMerchant category:")

print(
    transactions[
        "merchant_id"
    ]
    .map(
        merchants.set_index("merchant_id")["category"]
    )
    .value_counts()
)


print("\nFailure reasons:")

print(
    transaction_failures[
        "failure_reason"
    ].value_counts()
)


print("\nTotal transaction value:")

print(
    f"₹{transactions['amount'].sum():,.2f}"
)


print("\nAverage transaction value:")

print(
    f"₹{transactions['amount'].mean():,.2f}"
)


# ============================================================
# 12. RELATIONSHIP CHECKS
# ============================================================

print("\n" + "=" * 60)
print("RELATIONSHIP CHECKS")
print("=" * 60)


missing_customers = (
    ~transactions["customer_id"].isin(
        customers["customer_id"]
    )
).sum()

missing_merchants = (
    ~transactions["merchant_id"].isin(
        merchants["merchant_id"]
    )
).sum()

missing_failure_transactions = (
    ~transaction_failures["transaction_id"].isin(
        transactions["transaction_id"]
    )
).sum()


print(
    "Transactions with missing customer :",
    missing_customers
)

print(
    "Transactions with missing merchant :",
    missing_merchants
)

print(
    "Failure records with invalid txn   :",
    missing_failure_transactions
)


# ============================================================
# 13. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 60)
print("DATA GENERATION COMPLETE")
print("=" * 60)

print(
    f"\nFiles created inside: {OUTPUT_DIR}/"
)

print("""
1. customers.csv
2. merchants.csv
3. transactions.csv
4. transaction_failures.csv

The dataset is synthetic and intended for:
SQL Server + Power BI analytics.

Suggested next step:
Load the four CSV files into SQL Server and create
primary-key / foreign-key relationships.
""")

Generating customers...
Generating merchants...
Generating transactions...
Generating failure records...

Saving files...

DATASET SUMMARY
Customers              : 20,000
Merchants              : 2,000
Transactions           : 300,000
Failure records        : 13,957

Transaction status:
transaction_status
Success    282566
Failed      13957
Pending      3477
Name: count, dtype: int64

Payment method:
payment_method
UPI            134512
Debit Card      50000
Credit Card     48432
Net Banking     33656
Wallet          33400
Name: count, dtype: int64

Merchant category:
merchant_id
Grocery          103419
Food              70775
Utilities         27140
Fashion           22836
Other             16702
Entertainment     15428
Healthcare        14908
Travel            12204
Electronics        9884
Education          6704
Name: count, dtype: int64

Failure reasons:
failure_reason
Bank Server Error        3089
Insufficient Balance     2694
Payment Gateway Error    2314
Limit Exceeded          